In [1]:
# Carico la cartella one drive in cui ci sono:
# cartella video nutrition estimation / video annotation nutrition estimation
# cartella video nutrition change / video annotation nutrition change
# cartella video image nutrition estimation/ video annotation image estimation

# come ho scaricato i video?
# partendo dai file json ho lanciato lo script downloader sugli id estratti
# dei video scaricati ho fatto un samplig a N=128 FRAME
# caricato su one drive ed adesso linko da questo notebook
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Verifico la scheda video**
- Colab con GPU **L4** (24 GB di VRAM, circa 22 utilizzabili)
- serve Python 3.12 (vedi il controllo nella cella sotto)


In [ ]:
# Runtime: Runtime > Cambia tipo di runtime > GPU L4
import os, sys, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # meno frammentazione della VRAM

!nvidia-smi
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "Nessuna GPU: Runtime > Cambia tipo di runtime > L4"
print("GPU:", gpu, "| Python:", sys.version.split()[0])
if "L4" not in gpu:
    print("ATTENZIONE: il notebook e' tarato su L4. Un'altra GPU va bene se ha almeno 24 GB.")
# Serve Python 3.12, come il baseline: su 3.13 tokenizers 0.19.1 non ha la wheel e la
# compilazione con Rust fallisce, quindi pip non installa transformers 4.40 ne' decord.
assert sys.version_info[:2] == (3, 12), (
    f"Python {sys.version.split()[0]}: serve 3.12 (come il baseline). Runtime > Cambia tipo di runtime > "
    "Versione runtime: sceglierne una con Python 3.12, poi riavviare la sessione.")

In [ ]:
# PyTorch 2.7.0, come il baseline. torchaudio va allineato: quello preinstallato da Colab
# e' compilato per un altro torch e non si potrebbe importare.
%pip install -q torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0 --index-url https://download.pytorch.org/whl/cu128

In [ ]:
# Repo ufficiale di VideoLLaMA 2, copia pulita fissata a un commit: se il repo cambia,
# il notebook usa comunque lo stesso codice del baseline.
import os, shutil, sys
VL2_DIR = "/content/VideoLLaMA2"
VL2_COMMIT = "c0bb03abf6b8a6b9a8dccac006fb4db5d4d9e414"   # ultimo commit del repo (gennaio 2025)

%cd /content
if os.path.exists(VL2_DIR):
    shutil.rmtree(VL2_DIR)          # toglie anche le patch di una sessione precedente
!git clone -q https://github.com/DAMO-NLP-SG/VideoLLaMA2.git {VL2_DIR}
!git -C {VL2_DIR} checkout -q {VL2_COMMIT}
!git -C {VL2_DIR} log -1 --format="VideoLLaMA2: commit %h del %cd"

if VL2_DIR not in sys.path:
    sys.path.insert(0, VL2_DIR)

In [ ]:
%cd /content/VideoLLaMA2

In [ ]:
%pip install -q --upgrade pip setuptools wheel   # servono a compilare tokenizers se Python e' 3.13

In [ ]:
# Versioni ufficiali di VideoLLaMA2 (pyproject.toml del repo):
#   transformers==4.40.0, tokenizers==0.19.1, accelerate==0.26.1, decord==0.6.0
# - tokenizers 0.19.1 ha la wheel per Python 3.12 ma non per 3.13: li' si compila con Rust (alcuni minuti).
# - huggingface_hub < 1.0 e' richiesto da transformers 4.40 (Colab ha la 1.x).
# - timm < 1.1: il repo importa timm.models.layers, deprecato.
# - peft NON si installa: VideoLLaMA2 non lo usa in inferenza, e una versione nuova
#   potrebbe trascinare un aggiornamento di transformers.
# Gli avvisi rossi di pip su gradio, sentence-transformers, peft o torchao sono attesi:
# sono pacchetti di Colab che questo notebook non importa.
#
# %pip non si ferma se l'installazione fallisce: se un pacchetto (es. tokenizers) non si
# compila, pip annulla TUTTO il comando e restano le versioni preinstallate di Colab.
# Per questo si usa pip() qui sotto, che si ferma e mostra l'errore vero.
import sys, os, subprocess

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", *args],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("\n".join((r.stdout + r.stderr).splitlines()[-60:]))
        raise RuntimeError(f"pip install fallito: {' '.join(args)} (errore qui sopra)")

print("Python:", sys.version)

if sys.version_info >= (3, 13):
    print("Python 3.13: compilo tokenizers 0.19.1 con Rust (alcuni minuti)...")
    !curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
    os.environ["PATH"] += ":/root/.cargo/bin"
    os.environ["PYO3_USE_ABI3_FORWARD_COMPATIBILITY"] = "1"   # pyo3 di tokenizers 0.19 non conosce ancora 3.13
    pip("tokenizers==0.19.1", "--no-binary", "tokenizers")
else:
    pip("tokenizers==0.19.1")

pip("transformers==4.40.0", "tokenizers==0.19.1", "accelerate==0.26.1", "huggingface_hub>=0.23,<1.0",
    "timm>=1.0.3,<1.1", "einops", "decord==0.6.0", "av", "imageio", "opencv-python")

# Controllo: se le versioni non sono quelle del baseline ci si ferma qui
from importlib.metadata import version, PackageNotFoundError
def _v(k):
    try:
        return version(k)
    except PackageNotFoundError:
        return "NON INSTALLATO"
_att = {"torch": "2.7.0", "transformers": "4.40.0", "tokenizers": "0.19.1",
        "accelerate": "0.26.1", "decord": "0.6.0"}
_vers = {k: _v(k) for k in _att}
print(_vers)
_err = {k: v for k, v in _vers.items() if not v.startswith(_att[k])}
assert not _err, f"versioni errate {_err}, attese {_att}: controllare l'output di pip sopra"

In [8]:
# NON esiste una env var "TRANSFORMERS_ATTN_IMPLEMENTATION" in transformers.
# L'attention eager va richiesta passando attn_implementation="eager" a model_init
print("Eager attention verra' impostato via attn_implementation='eager' in model_init")


Eager attention verra' impostato via attn_implementation='eager' in model_init


In [ ]:
import os
# Evita il lookup del secret HF_TOKEN di Colab che va in timeout e "appende" il download
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"   # download piu' stabile
os.environ.pop("HF_TOKEN", None)

import gc, sys, pathlib, torch
# Se torch era gia' stato importato prima dell'installazione, in memoria c'e' ancora quello vecchio
assert torch.__version__.startswith("2.7.0"), (
    f"torch {torch.__version__} in memoria, atteso 2.7.0: Runtime > Riavvia sessione, poi rieseguire dall'inizio")
assert torch.cuda.is_available(), "CUDA non disponibile: il runtime non ha la GPU"

# Libero la VRAM da tentativi precedenti
for name in ['model', 'processor', 'tokenizer']:
    if name in dir():
        exec(f"del {name}")
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"VRAM libera: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GiB")

# -----------------------------------------------------------------------------
# PATCH encoder.py: il repo forza a mano config._attn_implementation="flash_attention_2"
# sulla CLIP/Siglip vision tower, ma questi modelli non supportano flash-attn -> ValueError.
# Sostituiamo con "eager" DIRETTAMENTE nel file sorgente, prima di importare videollama2.
# -----------------------------------------------------------------------------
_enc = pathlib.Path("/content/VideoLLaMA2/videollama2/model/encoder.py")
_txt = _enc.read_text()
if '"flash_attention_2"' in _txt:
    _enc.write_text(_txt.replace('"flash_attention_2"', '"eager"'))
    print("Patch applicata: flash_attention_2 -> eager in encoder.py")
else:
    print("Nessuna patch necessaria (flash_attention_2 non trovato in encoder.py)")

# Svuoto la cache dei moduli gia' importati, cosi' l'import qui sotto rilegge il file patchato
for _m in list(sys.modules):
    if _m.startswith("videollama2"):
        del sys.modules[_m]

# 1) Scarico lo snapshot VideoLLaMA2-7B (circa 16 GB)
from huggingface_hub import snapshot_download
model_path = snapshot_download("DAMO-NLP-SG/VideoLLaMA2-7B")
print("Modello scaricato in:", model_path)

# 2) Inizializzo passando il path LOCALE (niente altre chiamate di rete).
from videollama2 import model_init as videollama2_model_init

device = "cuda:0"
print("Inizio caricamento modello...")
model, processor, tokenizer = videollama2_model_init(
    model_path,
    device_map={"": device},
    attn_implementation="eager",   # forza eager sul modello linguistico
    use_flash_attn=False,
)

# imposto il modello in modalita' inferenza
model.eval()
print("Modello VideoLLaMA2-7B caricato con successo - dtype:", next(model.parameters()).dtype)
print("num_frames usati dal processor:", getattr(model.config, "num_frames", "default (8)"))
print(f"VRAM occupata: {torch.cuda.memory_allocated() / 1024**3:.1f} GiB "
      f"su {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")

---
# Parte 1 — Riproduzione della baseline HD-EPIC (VideoLLaMA 2 7B)

Obiettivo: verificare il valore riportato nel paper HD-EPIC (Tab. 2, colonna **Nutrition**)
per VideoLLaMA 2 7B, cioe' **32.7%**, contro un baseline random del 20%.

 **L'accuracy di categoria e' la media delle accuracy per prototipo**, non l'accuracy
   aggregata sulle domande. 

## 1.1 Motore di valutazione

Prompt, parsing delle risposte e metriche stanno in un unico modulo, condiviso con il
notebook di LLaVA-Video. E' questo che rende confrontabili i due modelli: se ogni
notebook avesse il suo parser, le differenze di accuracy misurerebbero anche le
differenze fra i parser.

La cella importa il modulo `src/hdepic/eval.py` del progetto, copiato su Drive da
`scripts/sync_drive.sh`. Non e' piu' incollato nel notebook: una sola copia per tutti.

In [ ]:
# Il motore di valutazione sta in src/hdepic/eval.py.
# Su Drive ce lo porta scripts/sync_drive.sh, lanciato dal Mac: prima di questa cella.
import os, sys, importlib
BASE = "/content/drive/MyDrive/Progetto Deep Learning 1"
SRC = os.path.join(BASE, "src")
assert os.path.exists(os.path.join(SRC, "hdepic", "eval.py")), (
    f"Manca {SRC}/hdepic/eval.py su Drive: lanciare scripts/sync_drive.sh dal Mac")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
from hdepic import eval as H
importlib.reload(H)   # se la cella viene rieseguita dopo una modifica del modulo


## 1.2 Percorsi e indicizzazione dei file

I video sono sparsi in piu' cartelle (`videos_*_sampled`, `videos_*`, `videos_*_panels`) e
lo stesso video puo' servire a piu' prototipi pur essendo stato scaricato in una sola
cartella. Per questo si costruisce **un indice globale** `video_id -> path` invece di
cercare in una cartella fissa per task: cercando per cartella si perdono domande che
sarebbero perfettamente eseguibili.

In [ ]:
import os, glob

BASE = "/content/drive/MyDrive/Progetto Deep Learning 1"
ANN_DIR    = os.path.join(BASE, "data", "hd-epic")                      # dove stanno i 3 json di annotation
OUT_DIR    = os.path.join(BASE, "results", "baseline")
FRAME_DIR  = os.path.join(BASE, "frames_image_nutrition_estimation")
CLIP_DIR   = os.path.join(BASE, "frames_nutrition_change")   # clip di nutrition_change (extract_clip_frames.py)

# tutte le cartelle di video e di pannelli presenti nel progetto
VIDEO_ROOTS = sorted(d for d in glob.glob(os.path.join(BASE, "*"))
                     if os.path.isdir(d) and "panel" not in os.path.basename(d).lower()
                     and os.path.basename(d).lower().startswith(("video", "videos")))
PANEL_ROOTS = sorted(d for d in glob.glob(os.path.join(BASE, "*"))
                     if os.path.isdir(d) and "panel" in os.path.basename(d).lower())

print("cartelle video :", [os.path.basename(d) for d in VIDEO_ROOTS])
print("cartelle panels:", [os.path.basename(d) for d in PANEL_ROOTS])

VIDEO_INDEX = H.build_video_index(VIDEO_ROOTS)
PANEL_INDEX = H.build_panels_index(PANEL_ROOTS)
print(f"\nindicizzati: {len(VIDEO_INDEX['sampled'])} video campionati, "
      f"{len(VIDEO_INDEX['full'])} video integrali, {len(PANEL_INDEX)} cartelle di pannelli")
n_clip = len(glob.glob(os.path.join(CLIP_DIR, "*", "frame_000.jpg")))
print(f"clip di nutrition_change in {os.path.basename(CLIP_DIR)}: {n_clip} (attese 50)")

# controllo che le annotation ci siano davvero
for t, (fname, _) in H.TASK_FILES.items():
    p = os.path.join(ANN_DIR, fname)
    print(("  OK " if os.path.exists(p) else "  MANCA ") + fname)

## Quante domande sono davvero eseguibili

Nel dataset scaricato ci sono mp4 **da 0 byte** e mp4 **troncati**: esistono come file ma
non si aprono. Se non li si individua prima, ogni domanda che li usa produce un'eccezione
che finisce contata come risposta sbagliata, e l'accuracy risulta piu' bassa del vero.

Il pre-volo apre davvero ogni file e separa le domande eseguibili da quelle da escludere.
Le escluse **non entrano nel denominatore**: i risultati verranno riportati con il loro N.

In [ ]:
READABLE = {}   # cache: aprire i video da Drive e' lento, si controlla una volta sola

cov, dettagli = H.preflight(ANN_DIR, VIDEO_INDEX, cache=READABLE, frame_cache=FRAME_DIR,
                              clip_cache=CLIP_DIR)
print(cov.to_string(index=False))

rotti = sorted({v for task in dettagli.values() for ids in task.values() for v in ids})
if rotti:
    print(f"\nFile mancanti o non decodificabili ({len(rotti)}):")
    for v in rotti:
        print("   ", v)

## 1.4 Il modello

L'unica parte specifica di VideoLLaMA 2 e' la funzione `infer_fn`. Il `processor['video']`
del repo accetta sia il path di un mp4 sia una **lista di path di immagini** e in entrambi i
casi applica il preprocessing corretto per l'encoder CLIP (resize 336x336, normalizzazione
CLIP, padding `expand2square`). Costruire il tensore a mano con normalizzazioni sbagliate
produce output senza senso, quindi si passa sempre dal processor.

Da notare: il processor ricampiona a `model.config.num_frames` (8 di default) **solo** quando
riceve il path di un video. Una lista di immagini la carica tutta (fino a `MAX_FRAMES=32`),
quindi le liste lunghe (pannelli, fotogrammi delle clip di `nutrition_change`) vengono
sottocampionate uniformemente a 8 in `videollama2_infer`. Il campionamento a 128 frame fatto a
monte **non** arriva quindi tutto al modello.

In [ ]:
import numpy as np
from videollama2 import mm_infer

MAX_NEW_TOKENS = 32          # basta: la risposta attesa e' una lettera
USE_SYSTEM_PROMPT = None     # il paper non usa Systemp Prompt (se lo utilizzassimo le performance si alzano)
NUM_FRAMES = getattr(model.config, "num_frames", 8)   # elementi visivi per domanda (default 8)

def sottocampiona(paths, n=NUM_FRAMES):
    """Riduce una lista di immagini a n elementi uniformi (come _immagini in LLaVA).

    Serve perche' process_video ricampiona a num_frames SOLO quando riceve il path di un
    video: una lista di immagini la carica tutta e la tronca a MAX_FRAMES=32. Senza questo
    passaggio i 32 pannelli arriverebbero tutti al modello (4x i token visivi del baseline).
    Liste piu' corte di n (es. le 5 immagini di image_nutrition_estimation) restano invariate.
    """
    if len(paths) <= n:
        return paths
    sel = np.linspace(0, len(paths) - 1, n, dtype=int)
    return [paths[i] for i in sel]

def videollama2_infer(visual, prompt):
    """visual: path di un mp4, oppure lista di path di immagini (frame o pannelli)."""
    if isinstance(visual, list):
        visual = sottocampiona(visual)
    tensor = processor["video"](visual)
    # stesso numero di elementi visivi in ogni condizione (video, pannelli, immagini)
    assert tensor.shape[0] == NUM_FRAMES, f"attesi {NUM_FRAMES} elementi visivi, arrivati {tensor.shape[0]}"
    return mm_infer(tensor, prompt, model, tokenizer, modal="video",
                    do_sample=False,          # greedy: run deterministico e ripetibile
                    max_new_tokens=MAX_NEW_TOKENS)

print("num_frames usati dal processor:", NUM_FRAMES)
print("dtype:", next(model.parameters()).dtype)

## 1.6 Inferenza completa

I risultati vengono scritti su file **una domanda alla volta**: se Colab si disconnette,
basta rilanciare la cella e riparte da dove era arrivata invece di rifare tutto.

In [ ]:
baseline = H.run_all(list(H.TASK_FILES), ANN_DIR, VIDEO_INDEX, videollama2_infer,
                     out_dir=OUT_DIR, tag="videollama2_baseline_clip",   # nuovo tag: nutrition_change ora usa la clip
                     system=USE_SYSTEM_PROMPT,
                     frame_extractor=H.extract_frame, frame_cache=FRAME_DIR,
                     readable_cache=READABLE, clip_cache=CLIP_DIR)

print(f"\nCompletate {len(baseline)} domande")

In [17]:
rep_base = H.accuracy_report(baseline,
                             label="VideoLLaMA 2 7B - baseline (video)",
                             paper_ref="VideoLLaMA 2 7B",
                             ann_dir=ANN_DIR)#costruisco il report

  VideoLLaMA 2 7B - baseline (video)   (n = 200 domande valutate)
                            corrette  valutate  accuracy_%  su_totale
task                                                                 
image_nutrition_estimation        28       100        28.0        100
nutrition_change                   8        50        16.0         50
video_nutrition_estimation        33        50        66.0         50

  Nutrition, metrica del paper (media per prototipo) : 36.7%
  Nutrition, accuracy aggregata sulle domande        : 34.5%
  Baseline random                                    : 20.0%
  Paper, VideoLLaMA 2 7B             : 32.7%   (delta +4.0)

  Risposte non interpretate o in errore: 0/200 (0.0%)
  Come e' stata letta la risposta:
    {'leading_letter': 200}

  Accuracy per tipo di domanda:
                  corrette  valutate  accuracy_%
q_type                                          
calories                12        41        29.3
carbs                   20        35      

### Come leggere il bias posizionale

L'ultima tabella confronta **su quale lettera il modello mette le sue risposte** con la
distribuzione reale delle risposte corrette. Se il modello fosse solo impreciso, le due
distribuzioni si assomiglierebbero. Se invece concentra le risposte su A e B mentre il
ground truth e' uniforme, allora non sta rispondendo alla domanda: sta seguendo una
preferenza a priori sul formato. E' un modo per distinguere *"il modello sbaglia"* da
*"il modello non sta guardando il video"*, ed e' la spiegazione tipica di un'accuracy
**sotto** il 20% del random.

---
# Parte 2 — Video Panels (Visaul Prompting)

Secondo esperimento, basato su *Video Panels for Long Video Understanding. L'idea: invece di dare al modello 8 fotogrammi sparsi, gli si
danno 8 **immagini pannellate 2x2**, ognuna contenente 4 fotogrammi consecutivi. A parita'
di finestra di contesto il modello copre cosi' 32 fotogrammi invece di 8, scambiando
risoluzione spaziale con risoluzione temporale.

I pannelli sono stati generati a monte da `paper_mosaic_sampling/mosaic.py` (32 pannelli per
video, griglia 2x2, quindi 128 fotogrammi coperti).

**Ambito del confronto:** solo `video_nutrition_estimation` (`H.PANEL_TASKS`).
- `image_nutrition_estimation` resta fuori perche' il suo input sono 5 fotogrammi isolati da
  video diversi: non c'e' una sequenza temporale da impacchettare.
- `nutrition_change` resta fuori perche' il suo input e' una clip di 1-27 s. Il paper applica i
  pannelli solo sopra `gamma*C*alpha*beta` = 1 s x 128 = 128 s; sotto soglia ripiega sul
  campionamento normale, cioe' sullo stesso input del baseline.

In [ ]:
cov_p, dett_p = H.preflight(ANN_DIR, VIDEO_INDEX, tasks=H.PANEL_TASKS,
                            panels_index=PANEL_INDEX, input_mode="panels",
                            cache=READABLE)
print(cov_p.to_string(index=False))

# quanti pannelli per video, e quanti ne arrivano davvero al modello
if PANEL_INDEX:
    vid, pannelli = next(iter(PANEL_INDEX.items()))
    visti = len(sottocampiona(pannelli))
    print(f"\nesempio: {vid} -> {len(pannelli)} pannelli su disco")
    print(f"al modello ne arrivano {visti} (sottocampionati in videollama2_infer), "
          f"cioe' {visti * 4} fotogrammi originali contro gli {NUM_FRAMES} del baseline")

In [ ]:
panels_run = H.run_all(H.PANEL_TASKS, ANN_DIR, VIDEO_INDEX, videollama2_infer,
                       out_dir=OUT_DIR, tag="videollama2_panels_8",   # nuovo tag: il vecchio run usava 32 pannelli, non va ripreso
                       panels_index=PANEL_INDEX, input_mode="panels",
                       system=USE_SYSTEM_PROMPT, readable_cache=READABLE)

rep_pan = H.accuracy_report(panels_run,
                            label="VideoLLaMA 2 7B - Video Panels 2x2",
                            ann_dir=ANN_DIR)

## 2.1 Confronto

Il confronto va fatto **sulle stesse domande**: il baseline viene quindi ristretto a
`video_nutrition_estimation` e alle domande effettivamente valutate anche nel run con i pannelli.

In [ ]:
comuni = set(panels_run["key"]) & set(baseline["key"])
b = baseline[baseline["key"].isin(comuni)]
p = panels_run[panels_run["key"].isin(comuni)]
print(f"domande confrontate a parita' di input: {len(comuni)}\n")

tab = H.compare_runs({"baseline (video)": b, "panels 2x2": p},
                     "VideoLLaMA 2 7B: video liscio vs pannelli 2x2")

## 2.2 Salvataggio dei risultati

Tutto su CSV, una riga per domanda, con la risposta grezza del modello e il metodo con cui
e' stata interpretata: senza questi due campi un numero di accuracy non e' verificabile
da nessuno.

In [21]:
import pandas as pd

tutto = pd.concat([baseline.assign(run="baseline"),
                   panels_run.assign(run="panels")], ignore_index=True)
csv = os.path.join(OUT_DIR, "videollama2_risultati_completi.csv")
tutto.to_csv(csv, index=False)

riepilogo = pd.DataFrame({
    "run": ["baseline (video)", "panels 2x2"],
    "nutrition_media_prototipi_%": [rep_base["nutrition_paper"], rep_pan["nutrition_paper"]],
    "nutrition_aggregata_%":       [rep_base["nutrition_pooled"], rep_pan["nutrition_pooled"]],
    "domande_valutate":            [len(baseline), len(panels_run)],
})
riepilogo.to_csv(os.path.join(OUT_DIR, "videollama2_riepilogo.csv"), index=False)

print(riepilogo.to_string(index=False))
print(f"\nsalvato: {csv}")
print(f"paper, VideoLLaMA 2 7B su Nutrition: {H.PAPER_NUTRITION['VideoLLaMA 2 7B']}%  "
      f"| random: 20.0%")

             run  nutrition_media_prototipi_%  nutrition_aggregata_%  domande_valutate
baseline (video)                         36.7                   34.5               200
      panels 2x2                         41.2                   41.4                99

salvato: /content/drive/MyDrive/Progetto Deep Learning 1/risultati/videollama2_risultati_completi.csv
paper, VideoLLaMA 2 7B su Nutrition: 32.7%  | random: 20.0%
